# From SPARQL to a Network: JILA Knowledge Graph

This notebook is for **teaching**: it shows how to move from **semantic graph data** in a public triple store to a **network** you can analyse and visualise in Python.

**What you will do**

1. Connect to the [JILA SPARQL endpoint](https://jila.zb.uzh.ch/sparql) and run `SELECT` queries.
2. Map result rows to **edges** (`source`, `target`, optional attributes).
3. Build a **`networkx`** graph, then **resolve `rdfs:label` / `skos:prefLabel`** for its node URIs.
4. **Visualise** the graph interactively with **`ipysigma`**.
5. Sketch **refinement** steps (filtering, projections): you can extend this after inspecting the data.

**Prerequisite ideas** (no RDF store write access here): the Itten pipeline’s [`materialiseRelations.py`](https://github.com/swiss-art-research-net/itten-pipeline/blob/main/scripts/materialiseRelations.py) **materialises** derived triples from patterns in YAML; [`network.yml`](https://github.com/swiss-art-research-net/itten-pipeline/blob/main/scripts/definitions/network.yml) declares those patterns in CIDOC CRM style. In this notebook we **read** similar relationship paths with `SELECT` instead of inserting triples.

---

## Notebook map

| Section | Topic |
|--------|--------|
| **1** | Setup (endpoint, imports) |
| **2** | Example SPARQL queries (inspired by `network.yml`) |
| **3** | Reusable helper functions |
| **4** | Fetch results, build a `networkx` graph, and attach **RDF labels** |
| **5** | Visualise with `ipysigma` |
| **6** | Refine, project, export (starting points) |


## 1) Setup and dependencies

We use **`requests`** for HTTP (already in the project `requirements.txt`) and standard JSON parsing of SPARQL Results JSON. **`networkx`** holds the graph; **`ipysigma`** draws it in the notebook.

> If anything is missing in your environment:
>
> ```bash
> pip install requests networkx ipysigma pandas
> ```

In [32]:
# HTTP client: the SPARQL 1.1 protocol returns JSON for SELECT queries when you ask for it.
import json
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import urlsplit

import networkx as nx
import pandas as pd
import requests
from ipysigma import Sigma

# Public read-only endpoint for the JILA knowledge graph (no authentication in the browser UI either).
SPARQL_ENDPOINT = "https://jila.zb.uzh.ch/sparql"

# Default timeout so a slow query fails clearly instead of hanging indefinitely.
HTTP_TIMEOUT_S = 120

print("Ready: endpoint =", SPARQL_ENDPOINT)

Ready: endpoint = https://jila.zb.uzh.ch/sparql


## 2) Example SPARQL queries (inspired by `network.yml`)

In the Itten pipeline, **`network.yml`** describes *patterns* that relate a **domain** type to a **range** type (for example a curated holding to an entity). The materialisation script wraps those patterns in `INSERT … WHERE` updates.

For **analysis**, we only need **`SELECT`** rows that identify **pairs of URIs** and a **human-readable relation label**. Below, two patterns mirror relation ids from the YAML (names shortened for class):

- **`collection_depicts_entity`** → path `?collection crm:P62_depicts ?entity`.
- **`collection_refers_to_entity`** → path `?collection crm:P128_carries/crm:P67_refers_to ?entity`.

We `BIND` each branch to the same variable names (`?source`, `?target`, `?relation`) so one Python function can consume all rows.

**Labels:** node identifiers in the graph are **URIs**. For Sigma (and for students) we also fetch **`rdfs:label`** and, when present, **`skos:prefLabel`** from the same endpoint, then pick one string per resource (language preference: German, then English, then any).

In [33]:
# PREFIX block: CRM (edges) plus common vocabulary properties used for human-readable labels.
PREFIXES = """
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
"""

# Query A — corresponds to relation id `collection_depicts_entity` in network.yml.
# Semantics: a curated holding (collection) "depicts" some CRM entity (here often an actor URI in the data).
QUERY_COLLECTION_DEPICITS_ENTITY = (
    PREFIXES
    + """
    SELECT ?source ?target ?relation WHERE {
      ?collection crm:P62_depicts ?entity .
      BIND(?collection AS ?source)
      BIND(?entity AS ?target)
      BIND("crm:P62_depicts" AS ?relation)
    }
    LIMIT 800
"""
)

# Query B — corresponds to `collection_refers_to_entity` (carrying object refers to entity).
QUERY_COLLECTION_REFERS_TO_ENTITY = (
    PREFIXES
    + """
    SELECT ?source ?target ?relation WHERE {
      ?collection crm:P128_carries/crm:P67_refers_to ?entity .
      BIND(?collection AS ?source)
      BIND(?entity AS ?target)
      BIND("crm:P128/P67_refers_to" AS ?relation)
    }
    LIMIT 800
"""
)

# Optional: one UNION query is easier to fetch once; pedagogically you can also run A and B separately and concatenate rows.
QUERY_COMBINED_EDGES = (
    PREFIXES
    + """
    SELECT ?source ?target ?relation WHERE {
      {
        ?collection crm:P62_depicts ?entity .
        BIND(?collection AS ?source)
        BIND(?entity AS ?target)
        BIND("crm:P62_depicts" AS ?relation)
      }
      UNION
      {
        ?collection crm:P128_carries/crm:P67_refers_to ?entity .
        BIND(?collection AS ?source)
        BIND(?entity AS ?target)
        BIND("crm:P128/P67_refers_to" AS ?relation)
      }
    }
    LIMIT 1500
"""
)

## 3) Reusable functions

Keeping **HTTP + JSON parsing + graph construction** in functions keeps the teaching narrative readable and lets you **move this block to a `.py` module** later without changing the story.

In [34]:
def sparql_select(
    endpoint: str,
    query: str,
    timeout_s: float = HTTP_TIMEOUT_S,
) -> Dict[str, Any]:
    """
    Run a SPARQL SELECT against an endpoint and return the parsed JSON object.

    Uses POST + application/x-www-form-urlencoded, which is widely supported and avoids URL length limits.
    """
    # SPARQL 1.1 Graph Store / Query Protocol: form field "query" holds the request body.
    headers = {
        "Accept": "application/sparql-results+json",
        "Content-Type": "application/x-www-form-urlencoded",
    }
    response = requests.post(
        endpoint,
        data={"query": query},
        headers=headers,
        timeout=timeout_s,
    )
    # Surface server error bodies for teaching/debugging (e.g. syntax error in SELECT).
    response.raise_for_status()
    return response.json()


def binding_to_uri(binding: Dict[str, Any], var_name: str) -> Optional[str]:
    """
    Extract a URI string from one SPARQL binding for variable `var_name`, or None if unbound / not a URI.
    """
    cell = binding.get(var_name)
    if not cell:
        return None
    if cell.get("type") != "uri":
        return None
    return cell.get("value")


def binding_to_literal(binding: Dict[str, Any], var_name: str) -> Optional[str]:
    """Extract a plain literal string value if present."""
    cell = binding.get(var_name)
    if not cell:
        return None
    if cell.get("type") != "literal":
        return None
    return cell.get("value")


def sparql_json_to_edge_rows(
    doc: Dict[str, Any],
    source_var: str = "source",
    target_var: str = "target",
    relation_var: str = "relation",
) -> List[Dict[str, str]]:
    """
    Convert SPARQL JSON results into a list of dicts with keys source, target, relation.

    Rows with missing URI parts are skipped so the graph builder stays strict.
    """
    rows: List[Dict[str, str]] = []
    for binding in doc.get("results", {}).get("bindings", []):
        s = binding_to_uri(binding, source_var)
        t = binding_to_uri(binding, target_var)
        if not s or not t:
            continue
        # Relation may be a literal BIND or omitted; fall back to "unspecified".
        rel = binding_to_literal(binding, relation_var) or "unspecified"
        rows.append({"source": s, "target": t, "relation": rel})
    return rows


# --- Label lookup (SPARQL → one readable string per resource URI) -----------------

# Teaching default: prefer German UI strings, then English, then first available (sorted for stability).
LABEL_LANG_PRIORITY: Tuple[str, ...] = ("de", "en", "fr", "it", "")


def _chunked(items: List[str], size: int) -> List[List[str]]:
    """Split a list into slices of at most `size` elements (SPARQL VALUES blocks can get very long)."""
    return [items[i : i + size] for i in range(0, len(items), size)]


def build_resource_label_query(resource_uris: List[str]) -> str:
    """
    Build a SELECT that returns (?resource ?labelText ?labelLang) for rdfs:label and skos:prefLabel.

    `VALUES` restricts the scan to the URIs you already have as nodes in the network.
    """
    # SPARQL requires IRIs in angle brackets; JILA HTTPS URIs are safe to embed literally here.
    values_body = " ".join(f"<{u}>" for u in resource_uris)
    return (
        """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?resource ?labelText ?labelLang WHERE {
  VALUES ?resource { """
        + values_body
        + """ }
  {
    ?resource rdfs:label ?lit .
    BIND(STR(?lit) AS ?labelText)
    BIND(LANG(?lit) AS ?labelLang)
  }
  UNION
  {
    ?resource skos:prefLabel ?lit .
    BIND(STR(?lit) AS ?labelText)
    BIND(LANG(?lit) AS ?labelLang)
  }
}
"""
    )


def sparql_json_to_label_rows(
    doc: Dict[str, Any],
    resource_var: str = "resource",
    text_var: str = "labelText",
    lang_var: str = "labelLang",
) -> List[Tuple[str, str, str]]:
    """Parse SPARQL JSON into (resource_uri, label_text, lang_code) tuples (multiple per resource allowed)."""
    out: List[Tuple[str, str, str]] = []
    for binding in doc.get("results", {}).get("bindings", []):
        res = binding_to_uri(binding, resource_var)
        text = binding_to_literal(binding, text_var)
        lang = binding_to_literal(binding, lang_var) or ""
        if res and text:
            out.append((res, text, lang))
    return out


def merge_label_rows(rows: List[Tuple[str, str, str]]) -> Dict[str, str]:
    """
    For each resource URI, choose a single display label from possibly many literals.

    Preference order is `LABEL_LANG_PRIORITY`; ties are broken lexicographically on the text.
    """
    by_uri: Dict[str, List[Tuple[str, str]]] = {}
    for uri, text, lang in rows:
        by_uri.setdefault(uri, []).append((text, lang))

    rank = {code: i for i, code in enumerate(LABEL_LANG_PRIORITY)}
    default_rank = len(LABEL_LANG_PRIORITY)

    def score(lang: str) -> int:
        return rank.get(lang, default_rank)

    chosen: Dict[str, str] = {}
    for uri, candidates in by_uri.items():
        # Sort by (language priority, label text) and take the best row.
        best = sorted(candidates, key=lambda pair: (score(pair[1]), pair[0]))[0][0]
        chosen[uri] = best
    return chosen


def truncate_label(text: str, max_len: int = 56) -> str:
    """Keep Sigma labels readable: long titles become one line with an ellipsis."""
    text = text.replace("\n", " ").strip()
    if len(text) <= max_len:
        return text
    return text[: max_len - 1] + "…"


def fetch_resource_labels(
    endpoint: str,
    resource_uris: List[str],
    chunk_size: int = 300,
    timeout_s: float = HTTP_TIMEOUT_S,
) -> Dict[str, str]:
    """
    Query the endpoint in batches and return uri → label (best-effort; missing URIs omitted).

    Chunking keeps each HTTP POST comfortably below typical parser limits when thousands of nodes exist.
    """
    unique = sorted(set(resource_uris))
    merged_rows: List[Tuple[str, str, str]] = []
    for chunk in _chunked(unique, chunk_size):
        q = build_resource_label_query(chunk)
        doc = sparql_select(endpoint, q, timeout_s=timeout_s)
        merged_rows.extend(sparql_json_to_label_rows(doc))
    return {uri: truncate_label(lbl) for uri, lbl in merge_label_rows(merged_rows).items()}


def fallback_label_from_uri(node: str, max_tail_len: int = 48) -> str:
    """Last path segment of the URI (what we used before RDF labels were loaded)."""
    path = urlsplit(node).path or ""
    tail = path.rstrip("/").split("/")[-1] if path else node
    return tail[:max_tail_len] + ("…" if len(tail) > max_tail_len else "")


def resource_kind_from_uri(uri: str) -> str:
    """
    Very small heuristic: classify JILA resource URIs by path segment for colouring in Sigma.
    This is not ontology-complete — it is only for visual orientation in class.
    """
    path = urlsplit(uri).path or ""
    parts = [p for p in path.split("/") if p]
    if not parts:
        return "other"
    # Typical pattern: .../collection/id, .../actor/id, .../place/id
    return parts[0] if parts[0] in {"collection", "actor", "place", "concept"} else "other"


def edge_rows_to_multi_digraph(rows: List[Dict[str, str]]) -> nx.MultiDiGraph:
    """
    Build a directed multigraph: parallel edges keep distinct CRM paths between the same URIs.
    """
    G = nx.MultiDiGraph()
    for row in rows:
        G.add_edge(row["source"], row["target"], relation=row["relation"])
    return G


def collapse_multi_edges_for_visualisation(G: nx.MultiDiGraph) -> nx.Graph:
    """
    ipysigma is happiest with a simple Graph; merge parallel MultiDiGraph edges into one undirected edge
    with a comma-separated list of relation labels.
    """
    S = nx.Graph()
    for u, v, key, data in G.edges(keys=True, data=True):
        rel = data.get("relation", "")
        if S.has_edge(u, v):
            existing = S[u][v].get("relation", "")
            parts = [p for p in {existing, rel} if p]
            S[u][v]["relation"] = ", ".join(sorted(parts))
        else:
            S.add_edge(u, v, relation=rel)
    return S


def annotate_nodes_for_sigma(G: nx.Graph, labels: Optional[Dict[str, str]] = None) -> None:
    """
    Mutate G in place: add node attributes consumed by Sigma (label, kind, degree).

    If `labels` is provided (uri → string from `fetch_resource_labels`), those strings win;
    otherwise we fall back to the short URI tail so the notebook still runs before the label query cell.
    """
    degree_dict = dict(G.degree())
    labels = labels or {}
    for node in G.nodes:
        kind = resource_kind_from_uri(node)
        human = labels.get(node)
        G.nodes[node]["kind"] = kind
        G.nodes[node]["label"] = human if human else fallback_label_from_uri(node)
        G.nodes[node]["degree"] = degree_dict.get(node, 0)


print(
    "Defined:",
    sparql_select.__name__,
    ",",
    edge_rows_to_multi_digraph.__name__,
    ",",
    fetch_resource_labels.__name__,
    ", …",
)

Defined: sparql_select , edge_rows_to_multi_digraph , fetch_resource_labels , …


## 4) Fetch triple patterns and build `networkx` graphs

We execute the **combined** query once, turn bindings into **edge rows**, then build:

- **`G_multi`**: full directed multigraph (good for counting parallel relation types).
- **`G_vis`**: simplified undirected graph for **interactive layout** in the next section.

Then we run a **second SPARQL round-trip**: a batched `VALUES { … }` query that retrieves `rdfs:label` / `skos:prefLabel` literals for every node URI currently in `G_vis`, and we write the chosen string into the `label` node attribute before plotting.

In [35]:
# 1) Call the endpoint and parse JSON.
raw = sparql_select(SPARQL_ENDPOINT, QUERY_COMBINED_EDGES)

In [36]:
# 2) Normalise to edge records the rest of the notebook understands.
edge_rows = sparql_json_to_edge_rows(raw)

In [37]:
# 3) Tabular view helps students inspect a sample before thinking in graph terms.
df_edges = pd.DataFrame(edge_rows)
print(f"Retrieved {len(df_edges):,} edges from SPARQL.")
# Print a text table so the preview appears even though this cell continues below.
print(df_edges.head(10).to_string(index=False))

Retrieved 1,500 edges from SPARQL.
                                                                          source                                                                     target               relation
https://resource.jila.zb.uzh.ch/collection/jila-0c0583bb7ac742b08269229ac595be7b https://resource.jila.zb.uzh.ch/actor/D199202E-FFBF-3F77-BC44-22F4AFB6148A crm:P128/P67_refers_to
https://resource.jila.zb.uzh.ch/collection/jila-0c0583bb7ac742b08269229ac595be7b https://resource.jila.zb.uzh.ch/actor/D80A3A9D-EB6F-37E9-A104-76BF84B84E9D crm:P128/P67_refers_to
https://resource.jila.zb.uzh.ch/collection/jila-0c0583bb7ac742b08269229ac595be7b https://resource.jila.zb.uzh.ch/actor/D9547011-AB46-38C2-86B2-EA87B7A33E22 crm:P128/P67_refers_to
https://resource.jila.zb.uzh.ch/collection/jila-0c0583bb7ac742b08269229ac595be7b https://resource.jila.zb.uzh.ch/actor/D9751E24-C82D-342E-87BB-ED09C6DA6138 crm:P128/P67_refers_to
https://resource.jila.zb.uzh.ch/collection/jila-0c0583bb7ac742b0826922

In [38]:
# 4) Structural graph(s) — RDF labels are attached in the following subsection once we know all node URIs.
G_multi = edge_rows_to_multi_digraph(edge_rows)
G_vis = collapse_multi_edges_for_visualisation(G_multi)

print(
    f"MultiDiGraph: {G_multi.number_of_nodes():,} nodes, "
    f"{G_multi.number_of_edges():,} directed edge instances."
)
print(
    f"Collapsed graph for viz: {G_vis.number_of_nodes():,} nodes, "
    f"{G_vis.number_of_edges():,} undirected edges."
)

# 5) Quick sanity check: relation mix.
print("Edge relation mix (on collapsed graph):")
print(
    pd.Series(nx.get_edge_attributes(G_vis, "relation")).value_counts().head(10)
)

MultiDiGraph: 1,288 nodes, 1,500 directed edge instances.
Collapsed graph for viz: 1,288 nodes, 1,487 undirected edges.
Edge relation mix (on collapsed graph):
crm:P128/P67_refers_to                     1388
crm:P62_depicts                              86
crm:P128/P67_refers_to, crm:P62_depicts      13
Name: count, dtype: int64


### 4b) Human-readable labels via SPARQL

URIs are stable graph keys but poor **legends** on a plot. The public JILA store publishes `rdfs:label` (and sometimes `skos:prefLabel`) on many resources.

The helper `build_resource_label_query` emits a `SELECT` with a `VALUES ?resource { … }` block listing **only the nodes already in `G_vis`**, so the triple pattern scan stays bounded. Results are merged in Python with a simple **language priority** (see `LABEL_LANG_PRIORITY` in the functions cell).

In [39]:
# Every graph node is a full resource IRI — those are the subjects we ask the endpoint to label.
all_node_uris = list(G_vis.nodes())

# One batched round-trip per chunk (see `fetch_resource_labels` for the exact SPARQL shape).
resource_labels = fetch_resource_labels(SPARQL_ENDPOINT, all_node_uris)

# Refresh Sigma-facing attributes now that `resource_labels` is populated (falls back to URI tail if missing).
annotate_nodes_for_sigma(G_vis, labels=resource_labels)

n_labeled = sum(1 for u in all_node_uris if u in resource_labels)
print(
    f"Labeled {n_labeled:,} of {len(all_node_uris):,} nodes via rdfs:label / skos:prefLabel "
    f"({100.0 * n_labeled / max(len(all_node_uris), 1):.1f}% coverage)."
)

# Quick qualitative check: show a handful of resolved titles / names next to the opaque local ids.
print("Sample labels:")
for uri, lbl in list(resource_labels.items())[:15]:
    tail = uri.rstrip("/").split("/")[-1]
    print(f"  {tail}  →  {lbl}")

Labeled 1,288 of 1,288 nodes via rdfs:label / skos:prefLabel (100.0% coverage).
Sample labels:
  27041360-C9F8-33B8-AEEB-E212F6AB7D2F  →  Itten, Anneliese
  1D0286A8-BB44-3EB0-B4C3-9831D4F85736  →  Bauer in Flirsch, Gastgeber von Johannes Itten
  01FEE728-C562-3B73-9D5B-2DB4120DFA91  →  Eyquem, Eva
  0C9BC1DF-0D49-3F89-A781-B429C56CF725  →  Fischer, Franz
  0DC9288B-95FD-3C5C-A96E-F84DE6523132  →  Künstler in Stellungnahme zu einer schweizerischen Kuns…
  36AF4FA4-9C53-36C5-977F-857EE8EAE363  →  Henauer, Walter
  0FF86686-CC84-3054-B32D-FFD8B491BC89  →  Beuys, Joseph
  25B57361-2490-324E-9825-FBF38237F898  →  Schülerinnen und Schüler der Itten-Schule Berlin
  1B6B8845-9DED-3F59-B588-2ED1C712CECB  →  Slutzky, Naum
  1A552E68-8897-3FFD-B2F5-A18FC0A0B7B8  →  Stifter, Adalbert
  2BE7C15B-0414-3ACE-822F-E6D2AD099A76  →  Schaer, Adolf
  0D7C5E17-2827-3E33-BEF5-EA41E087D343  →  Kreutzfeldt, Hansjürgen
  5D40F3D1-6E71-3FC5-A577-13C32FA2DB36  →  Roth, Alfred
  08C9328A-ECA6-321B-B297-FA58DE6052

## 5) Visualise with `ipysigma`

We colour nodes by the coarse **`kind`** heuristic and size them by **degree** on the collapsed graph. **`node_label`** reads the **`label`** attribute, which section **4b** fills from the triple store where possible (otherwise the short URI tail). Pan, zoom, and hover behave like a small Gephi-in-the-notebook — useful for **qualitative** inspection before any heavy metrics.

In [40]:
# Interactive widget: requires a kernel that supports ipywidgets (standard in JupyterLab / VS Code).
Sigma(
    G_vis,
    node_color="kind",
    node_size="degree",
    node_label="label",
    #default_edge_type="curve",
    start_layout=True,
    clickable_edges=True,
    hide_info_panel=False,
)

Sigma(nx.Graph with 1,288 nodes and 1,487 edges)

## 6) Refine, project, export — starting points

After you inspect the live graph, you typically:

- **Filter** by node `kind`, degree, or specific URI prefixes.
- **Project** a bipartite graph (e.g. collections vs actors) onto one node set using `networkx.algorithms.bipartite`.
- **Export** to `.gexf` for Gephi Web / Cytoscape.

In [41]:
# Example refinement: keep only nodes that look like "collection" or "actor" in the URI heuristic.
# Adjust the allowed set once you decide on a subgraph story (e.g. only actors, only places).
ALLOWED_KINDS = {"collection", "actor"}

nodes_kept = {n for n, d in G_vis.nodes(data=True) if d.get("kind") in ALLOWED_KINDS}
G_refined = G_vis.subgraph(nodes_kept).copy()
# Reuse the same uri→label map built for `G_vis` so subgraph nodes keep human-readable titles.
annotate_nodes_for_sigma(G_refined, labels=resource_labels)

print(
    f"Refined graph: {G_refined.number_of_nodes():,} nodes, "
    f"{G_refined.number_of_edges():,} edges (kinds in {sorted(ALLOWED_KINDS)})."
)

# Uncomment to compare interactively:
# Sigma(G_refined, node_color="kind", node_size="degree", node_label="label", start_layout=True)

Refined graph: 1,224 nodes, 1,423 edges (kinds in ['actor', 'collection']).


In [42]:
# Example: bipartite sets inferred from URI heuristic — useful before a one-mode projection.
# If the heuristic does not match your ontology lesson, replace it with a SPARQL-driven node list.

def bipartite_node_sets_from_kind(G: nx.Graph) -> Tuple[set, set]:
    """
    Split nodes into two buckets (collection vs non-collection) for bipartite helpers.
    Returns (top_nodes, bottom_nodes) as sets of node identifiers (here: URI strings).
    """
    tops = {n for n, d in G.nodes(data=True) if d.get("kind") == "collection"}
    bottoms = {n for n, d in G.nodes(data=True) if d.get("kind") != "collection"}
    return tops, bottoms


top_nodes, bottom_nodes = bipartite_node_sets_from_kind(G_refined)
print(f"Top (collection): {len(top_nodes):,} | Bottom (other): {len(bottom_nodes):,}")

# Project onto "bottom" nodes: two bottom nodes are linked if they share a neighbour in `top_nodes`.
# Uncomment when `networkx` bipartite coverage is part of your lesson plan:
from networkx.algorithms import bipartite
G_projected = bipartite.weighted_projected_graph(G_refined, bottom_nodes, ratio=True)
print(G_projected.number_of_nodes(), G_projected.number_of_edges())

Top (collection): 431 | Bottom (other): 793
793 17206


In [43]:
# One-mode projection of `G_refined` onto non-collection nodes (`bottom_nodes` from the previous cell).
# Edge attribute `weight` counts how many collection neighbours two bottom nodes share (see `networkx` bipartite docs).
# Many weak ties make the widget slow, so we keep edges with weight at least the threshold below.
MIN_SHARED_COLLECTIONS = 0

G_projected_vis = nx.Graph(
    (u, v, d)
    for u, v, d in G_projected.edges(data=True)
    if d.get("weight", 0) >= MIN_SHARED_COLLECTIONS
)
annotate_nodes_for_sigma(G_projected_vis, labels=resource_labels)

print(
    f"Projected view: {G_projected_vis.number_of_nodes():,} nodes, "
    f"{G_projected_vis.number_of_edges():,} edges "
    f"(weight ≥ {MIN_SHARED_COLLECTIONS})."
)

Projected view: 718 nodes, 17,206 edges (weight ≥ 0).


In [44]:
# Every node in the projected view: display label and degree (co-mention count in this graph).
# Sorted by degree descending so hubs appear first — useful before / alongside the Sigma layout.
_proj_degree = dict(G_projected_vis.degree())
_df_proj_nodes = (
    pd.DataFrame(
        [
            {
                "label": G_projected_vis.nodes[n].get("label", ""),
                "degree": _proj_degree.get(n, 0),
            }
            for n in G_projected_vis.nodes()
        ]
    )
    .sort_values("degree", ascending=False)
    .reset_index(drop=True)
)

print(_df_proj_nodes.to_string(index=False))

                                                   label  degree
                                       Vecellio, Tiziano     209
                             Goethe, Johann Wolfgang von     206
                              Bruegel, Pieter, de Oudere     190
                                              Klee, Paul     190
                                           Cézanne, Paul     189
                                         Itten, Johannes     188
                                            Witz, Konrad     188
                                      Giotto, di Bondone     184
                             Pellegrini, Alfred Heinrich     177
                                                El Greco     166
                                           Hölzel, Adolf     163
                                       Gogh, Vincent van     161
                                          Picasso, Pablo     155
                   Velázquez, Diego Rodríguez de Silva y     155
                         